# PQID Metadata Design And Evaluation

This notebook creates an **additive metadata-design layer** between corpus enrichment and downstream seed generation / training.

The goal is not to inflate the schema with generic labels. The goal is to add a disciplined set of **audit-grade provenance, governance, split, and transparency fields** that materially help with:
- release governance
- split integrity and leakage control
- reproducibility and contamination discussion
- later training analysis and paper claims
- XAI-style interpretability of when the model should generate, repair, diagnose, or hedge

This notebook does **not** replace or rewrite the upstream corpus. It writes:
- a JSONL sidecar overlay containing only the new metadata-design fields
- a merged JSONL view that preserves the original records while adding the new fields into `metadata`
- a JSON and Markdown evaluation report with coverage tables, field distributions, cross-tabs, split-group statistics, and near-duplicate-group statistics
- a separate license-governance audit report for release-facing analysis

Current transparency-centered additions in `metadata_design_v3` include:
- `source_snapshot_timestamp`, `source_snapshot_granularity`, and `source_revision_id` for provenance traceability
- `license_evidence_source`, `license_detection_method`, and `release_view_membership` for governance interpretability
- `lineage_parent_id` for stable parent-pointer lineage across later seed/paraphrase artifacts
- `benchmark_view_membership` for direct benchmark-packaging analysis
- `near_duplicate_group_id` for leakage-aware analysis beyond source lineage
- `domain_slice` and `shift_axis` for subgroup and robustness analysis
- `review_trace_id` for audit-trace reconstruction
- `permission_response_status` and `manual_license_review_status` for explicit governance-workflow state


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "PQID").exists() and (candidate / "PQID" / "data").exists():
            return candidate
        if candidate.name == "PQID" and (candidate / "data").exists() and (candidate / "scripts").exists():
            return candidate.parent
    raise RuntimeError("Could not locate repo root from the current notebook working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
PQID_ROOT = REPO_ROOT / "PQID"
PROCESSED_DIR = PQID_ROOT / "data" / "processed"
SCRIPT_DIR = PQID_ROOT / "scripts" / "04_metadata_analysis"

DERIVE_SCRIPT = SCRIPT_DIR / "derive_pqid_metadata_design_fields.py"
EVALUATE_SCRIPT = SCRIPT_DIR / "evaluate_pqid_metadata_design_fields.py"
LICENSE_AUDIT_SCRIPT = SCRIPT_DIR / "audit_pqid_license_governance.py"

BASE_INPUT_FILE = PROCESSED_DIR / "pqid_2026_enriched_github_circuits.jsonl"
MASTER_OVERLAY_FILE = PROCESSED_DIR / "pqid_2026_master_corpus.jsonl"

# Use "smoke_test" to write isolated artifacts and cap rows by default.
# Use "full" to write the canonical full-corpus outputs.
METADATA_DESIGN_RUN_MODE = "full"

# In smoke-test mode, leaving this as None automatically uses 200 rows.
# In full mode, leave this as None to process the full corpus.
METADATA_DESIGN_MAX_ROWS = None

if METADATA_DESIGN_RUN_MODE not in {"full", "smoke_test"}:
    raise ValueError("METADATA_DESIGN_RUN_MODE must be 'full' or 'smoke_test'.")

if METADATA_DESIGN_RUN_MODE == "smoke_test":
    if METADATA_DESIGN_MAX_ROWS is None:
        METADATA_DESIGN_MAX_ROWS = 200
    OVERLAY_OUTPUT_FILE = PROCESSED_DIR / "pqid_2026_metadata_design_overlay_v3_smoke.jsonl"
    MERGED_OUTPUT_FILE = PROCESSED_DIR / "pqid_2026_enriched_github_circuits_plus_metadata_design_v3_smoke.jsonl"
    EVAL_REPORT_JSON_FILE = PROCESSED_DIR / "pqid_metadata_design_evaluation_report_v3_smoke.json"
    EVAL_REPORT_MD_FILE = PROCESSED_DIR / "pqid_metadata_design_evaluation_report_v3_smoke.md"
    LICENSE_REPORT_JSON_FILE = PROCESSED_DIR / "pqid_license_governance_report_v3_smoke.json"
    LICENSE_REPORT_MD_FILE = PROCESSED_DIR / "pqid_license_governance_report_v3_smoke.md"
else:
    OVERLAY_OUTPUT_FILE = PROCESSED_DIR / "pqid_2026_metadata_design_overlay_v3.jsonl"
    MERGED_OUTPUT_FILE = PROCESSED_DIR / "pqid_2026_enriched_github_circuits_plus_metadata_design_v3.jsonl"
    EVAL_REPORT_JSON_FILE = PROCESSED_DIR / "pqid_metadata_design_evaluation_report_v3.json"
    EVAL_REPORT_MD_FILE = PROCESSED_DIR / "pqid_metadata_design_evaluation_report_v3.md"
    LICENSE_REPORT_JSON_FILE = PROCESSED_DIR / "pqid_license_governance_report_v3.json"
    LICENSE_REPORT_MD_FILE = PROCESSED_DIR / "pqid_license_governance_report_v3.md"


def run_command(command: list[str]) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        command,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    if result.stdout.strip():
        print(result.stdout.rstrip())
    if result.stderr.strip():
        print("stderr:\n" + result.stderr.rstrip())
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}: {' '.join(command)}")
    return result


def markdown_table(headers: list[str], rows: list[list[object]]) -> str:
    rows_as_text = [[str(cell) for cell in row] for row in rows]
    widths = [len(header) for header in headers]
    for row in rows_as_text:
        for index, cell in enumerate(row):
            widths[index] = max(widths[index], len(cell))

    def fmt(values: list[str]) -> str:
        return "| " + " | ".join(value.ljust(widths[index]) for index, value in enumerate(values)) + " |"

    divider = "| " + " | ".join("-" * width for width in widths) + " |"
    return "\n".join([fmt(headers), divider, *(fmt(row) for row in rows_as_text)])


print("run mode:", METADATA_DESIGN_RUN_MODE)
print("max rows:", METADATA_DESIGN_MAX_ROWS if METADATA_DESIGN_MAX_ROWS is not None else "full corpus")
print("repo root:", REPO_ROOT)
print("base input:", BASE_INPUT_FILE)
print("master overlay:", MASTER_OVERLAY_FILE)
print("overlay output:", OVERLAY_OUTPUT_FILE)
print("merged output:", MERGED_OUTPUT_FILE)
print("evaluation json:", EVAL_REPORT_JSON_FILE)
print("license audit json:", LICENSE_REPORT_JSON_FILE)


## Stage M0 — Choose Full Run Or Smoke Test

Use the setup cell above as the control point for reproducible testing.

- Set `METADATA_DESIGN_RUN_MODE = "smoke_test"` to write isolated `*_smoke.*` artifacts.
- Leave `METADATA_DESIGN_MAX_ROWS = None` in smoke-test mode to automatically run on `200` rows.
- Set `METADATA_DESIGN_RUN_MODE = "full"` to write the canonical full-corpus outputs.

Recommended audit path before trusting the notebook end to end:
1. Run the setup cell in `smoke_test` mode.
2. Run `Stage M1`, `Stage M2`, `Stage M3`, `Stage M4`, `Stage M5`, and `Stage M6`.
3. Verify that the smoke-test JSONL and report files were created and that the preview tables render correctly.
4. Switch back to `full` mode only after the smoke-test run looks correct.


## Stage M1 — Derive Additive Metadata-Design Fields

This stage materializes the **row-level metadata overlay**. It builds two artifacts:
- `pqid_2026_metadata_design_overlay_v3.jsonl` — sidecar overlay with only the new fields
- `pqid_2026_enriched_github_circuits_plus_metadata_design_v3.jsonl` — merged view with the new fields written into `metadata`

The derivation uses the full enriched corpus and overlays benchmark-ready master-corpus fields where available before computing the new metadata-design fields.

This version adds the remaining schema-tightening fields that were still missing after `metadata_design_v2`, including:
- provenance semantics: `source_snapshot_granularity`
- lineage: `lineage_parent_id`
- benchmark packaging: `benchmark_view_membership`
- governance workflow state: `permission_response_status`, `manual_license_review_status`


In [ ]:
command = [
    sys.executable,
    str(DERIVE_SCRIPT),
    "--input-file", str(BASE_INPUT_FILE),
    "--master-overlay-file", str(MASTER_OVERLAY_FILE),
    "--overlay-output-file", str(OVERLAY_OUTPUT_FILE),
    "--merged-output-file", str(MERGED_OUTPUT_FILE),
]
if METADATA_DESIGN_MAX_ROWS:
    command.extend(["--max-rows", str(METADATA_DESIGN_MAX_ROWS)])
run_command(command)


## Stage M2 — Evaluate Metadata-Design Fields

This stage produces a machine-readable JSON report and a Markdown report with:
- field coverage
- field distributions for the behavioral, provenance, and governance additions
- cross-tabs against existing validation, benchmark, and licensing signals
- split-group statistics for provenance-aware train / validation / test construction
- near-duplicate-group statistics for leakage-aware analysis beyond source lineage

The purpose of this stage is to answer two questions clearly:
1. Were the new fields actually populated on the corpus?
2. Do their distributions and correlations look semantically plausible enough to support later publication claims?

At `metadata_design_v3`, we especially want to inspect whether the new lineage, benchmark-membership, and governance-workflow fields are internally coherent rather than merely non-empty.


In [ ]:
command = [
    sys.executable,
    str(EVALUATE_SCRIPT),
    "--input-file", str(MERGED_OUTPUT_FILE),
    "--report-json-file", str(EVAL_REPORT_JSON_FILE),
    "--report-md-file", str(EVAL_REPORT_MD_FILE),
]
if METADATA_DESIGN_MAX_ROWS:
    command.extend(["--max-rows", str(METADATA_DESIGN_MAX_ROWS)])
run_command(command)


## Stage M3 — Preview The Evaluation Tables

This cell reopens the JSON report and prints a concise notebook-side preview so the result is legible without opening the raw files.


In [ ]:
report = json.loads(EVAL_REPORT_JSON_FILE.read_text(encoding="utf-8"))

coverage_rows = []
for field, missing in report["field_missing_counts"].items():
    coverage_rows.append([field, report["rows"] - missing, missing])
display(Markdown("### Field Coverage\n\n" + markdown_table(["field", "present_rows", "missing_rows"], coverage_rows)))

summary_rows = []
for field in [
    "source_snapshot_timestamp",
    "source_snapshot_granularity",
    "license_evidence_source",
    "license_detection_method",
    "release_view_membership",
    "benchmark_view_membership",
    "expected_model_stance",
    "context_sufficiency_class",
    "repairability_band",
    "evidence_regime",
    "split_group_source",
    "domain_slice",
    "shift_axis",
    "permission_response_status",
    "manual_license_review_status",
]:
    distribution = report["field_value_counts"].get(field, {})
    for value, count in distribution.items():
        summary_rows.append([field, value, count])
display(Markdown("### Field Distributions\n\n" + markdown_table(["field", "value", "count"], summary_rows)))

split_stats = report["split_group_stats"]
display(Markdown(
    "### Split-Group Statistics\n\n" + markdown_table(
        ["metric", "value"],
        [
            ["unique_groups", split_stats["unique_groups"]],
            ["singleton_groups", split_stats["singleton_groups"]],
            ["non_singleton_groups", split_stats["non_singleton_groups"]],
            ["max_group_size", split_stats["max_group_size"]],
            ["avg_group_size", split_stats["avg_group_size"]],
            ["median_group_size", split_stats["median_group_size"]],
        ],
    )
))

near_dup_stats = report["near_duplicate_group_stats"]
display(Markdown(
    "### Near-Duplicate Group Statistics\n\n" + markdown_table(
        ["metric", "value"],
        [
            ["unique_groups", near_dup_stats["unique_groups"]],
            ["singleton_groups", near_dup_stats["singleton_groups"]],
            ["non_singleton_groups", near_dup_stats["non_singleton_groups"]],
            ["max_group_size", near_dup_stats["max_group_size"]],
            ["avg_group_size", near_dup_stats["avg_group_size"]],
            ["median_group_size", near_dup_stats["median_group_size"]],
        ],
    )
))

for table_name in [
    "expected_model_stance__by_validation_status",
    "expected_model_stance__by_benchmark_suitability_tier_v2",
    "repairability_band__by_expected_model_stance",
    "license_evidence_source__by_license_category",
    "license_detection_method__by_license_category",
    "release_view_membership__by_distribution_rights_status",
    "benchmark_view_membership__by_expected_model_stance",
    "domain_slice__by_expected_model_stance",
    "shift_axis__by_expected_model_stance",
    "permission_response_status__by_distribution_rights_status",
    "manual_license_review_status__by_distribution_rights_status",
    "source_snapshot_granularity__by_license_evidence_source",
]:
    table = report["cross_tabs"][table_name]
    columns = sorted({column for row in table.values() for column in row})
    rows = []
    for row_key, row_values in table.items():
        rows.append([row_key, *[row_values.get(column, 0) for column in columns]])
    display(Markdown(f"### {table_name}\n\n" + markdown_table(["row_key", *columns], rows)))


## Stage M4 — Inspect Representative Records

This local audit samples a few records from the merged view so we can verify that the new metadata fields look semantically plausible in context.

The point is not only to check the behavior-oriented fields like `expected_model_stance`, but also to inspect the new provenance, lineage, benchmark-view, and governance additions together in one row-level view.


In [ ]:
from collections import defaultdict

examples = defaultdict(list)
target_stances = {"generate", "repair", "diagnose", "robustness_compare"}

with MERGED_OUTPUT_FILE.open(encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        metadata = row.get("metadata", {})
        stance = metadata.get("expected_model_stance")
        if stance in target_stances and len(examples[stance]) < 2:
            examples[stance].append({
                "file_path": metadata.get("file_path"),
                "validation_status": metadata.get("validation_status"),
                "benchmark_suitability_tier_v2": metadata.get("benchmark_suitability_tier_v2"),
                "hallucination_type": metadata.get("hallucination_type"),
                "source_snapshot_timestamp": metadata.get("source_snapshot_timestamp"),
                "source_snapshot_granularity": metadata.get("source_snapshot_granularity"),
                "source_revision_id": metadata.get("source_revision_id"),
                "license_evidence_source": metadata.get("license_evidence_source"),
                "license_detection_method": metadata.get("license_detection_method"),
                "release_view_membership": metadata.get("release_view_membership"),
                "lineage_parent_id": metadata.get("lineage_parent_id"),
                "benchmark_view_membership": metadata.get("benchmark_view_membership"),
                "expected_model_stance": metadata.get("expected_model_stance"),
                "context_sufficiency_class": metadata.get("context_sufficiency_class"),
                "repairability_band": metadata.get("repairability_band"),
                "evidence_regime": metadata.get("evidence_regime"),
                "split_group_source": metadata.get("split_group_source"),
                "near_duplicate_group_id": metadata.get("near_duplicate_group_id"),
                "domain_slice": metadata.get("domain_slice"),
                "shift_axis": metadata.get("shift_axis"),
                "review_trace_id": metadata.get("review_trace_id"),
                "permission_response_status": metadata.get("permission_response_status"),
                "manual_license_review_status": metadata.get("manual_license_review_status"),
            })
        if all(len(examples[stance]) >= 2 for stance in target_stances):
            break

for stance in sorted(examples):
    display(Markdown(f"### {stance}"))
    for example in examples[stance]:
        print(json.dumps(example, ensure_ascii=False, indent=2))


## Stage M5 — Audit License Governance And Release Buckets

This stage runs a dedicated governance audit over the merged corpus. It does not remove rows. Instead, it turns the license metadata plus the new transparency fields into release-usable governance summaries and identifies the unresolved repositories that matter most for outreach or restricted-release handling.

At `metadata_design_v3`, this stage also surfaces the newly added workflow-state fields `permission_response_status` and `manual_license_review_status`, so the governance story is no longer only inferential.


In [ ]:
from pathlib import Path

def _metadata_notebook_bootstrap_license_stage():
    def _find_repo_root(start: Path) -> Path:
        for candidate in [start, *start.parents]:
            if (candidate / "PQID").exists():
                return candidate
        raise FileNotFoundError("Could not locate the PQID repository root from the current working directory.")

    globals_dict = globals()
    if "METADATA_DESIGN_RUN_MODE" not in globals_dict:
        globals_dict["METADATA_DESIGN_RUN_MODE"] = "full"
    if "METADATA_DESIGN_MAX_ROWS" not in globals_dict:
        globals_dict["METADATA_DESIGN_MAX_ROWS"] = None

    repo_root = globals_dict.get("REPO_ROOT")
    if repo_root is None:
        repo_root = _find_repo_root(Path.cwd())
        globals_dict["REPO_ROOT"] = repo_root

    pqid_root = globals_dict.get("PQID_ROOT") or (repo_root / "PQID")
    processed_dir = globals_dict.get("PROCESSED_DIR") or (pqid_root / "data" / "processed")
    script_dir = globals_dict.get("SCRIPT_DIR") or (pqid_root / "scripts" / "04_metadata_analysis")
    globals_dict["PQID_ROOT"] = pqid_root
    globals_dict["PROCESSED_DIR"] = processed_dir
    globals_dict["SCRIPT_DIR"] = script_dir
    globals_dict["LICENSE_AUDIT_SCRIPT"] = script_dir / "audit_pqid_license_governance.py"

    if globals_dict["METADATA_DESIGN_RUN_MODE"] == "smoke_test":
        globals_dict["MERGED_OUTPUT_FILE"] = processed_dir / "pqid_2026_enriched_github_circuits_plus_metadata_design_v3_smoke.jsonl"
        globals_dict["LICENSE_REPORT_JSON_FILE"] = processed_dir / "pqid_license_governance_report_v3_smoke.json"
        globals_dict["LICENSE_REPORT_MD_FILE"] = processed_dir / "pqid_license_governance_report_v3_smoke.md"
    else:
        globals_dict["MERGED_OUTPUT_FILE"] = processed_dir / "pqid_2026_enriched_github_circuits_plus_metadata_design_v3.jsonl"
        globals_dict["LICENSE_REPORT_JSON_FILE"] = processed_dir / "pqid_license_governance_report_v3.json"
        globals_dict["LICENSE_REPORT_MD_FILE"] = processed_dir / "pqid_license_governance_report_v3.md"

_metadata_notebook_bootstrap_license_stage()

command = [
    sys.executable,
    str(LICENSE_AUDIT_SCRIPT),
    "--input-file", str(MERGED_OUTPUT_FILE),
    "--report-json-file", str(LICENSE_REPORT_JSON_FILE),
    "--report-md-file", str(LICENSE_REPORT_MD_FILE),
]
if METADATA_DESIGN_MAX_ROWS:
    command.extend(["--max-rows", str(METADATA_DESIGN_MAX_ROWS)])
run_command(command)


## Stage M6 — Preview License Governance Tables

This cell reopens the license-governance report and prints the release-bucket summary plus the highest-priority unresolved repositories.

It also shows the newly added workflow-state counts so the notebook distinguishes unresolved rights status from actual outreach/review progress.


In [ ]:
from pathlib import Path

def _metadata_notebook_bootstrap_license_preview():
    def _find_repo_root(start: Path) -> Path:
        for candidate in [start, *start.parents]:
            if (candidate / "PQID").exists():
                return candidate
        raise FileNotFoundError("Could not locate the PQID repository root from the current working directory.")

    globals_dict = globals()
    if "METADATA_DESIGN_RUN_MODE" not in globals_dict:
        globals_dict["METADATA_DESIGN_RUN_MODE"] = "full"

    repo_root = globals_dict.get("REPO_ROOT")
    if repo_root is None:
        repo_root = _find_repo_root(Path.cwd())
        globals_dict["REPO_ROOT"] = repo_root

    pqid_root = globals_dict.get("PQID_ROOT") or (repo_root / "PQID")
    processed_dir = globals_dict.get("PROCESSED_DIR") or (pqid_root / "data" / "processed")
    globals_dict["PQID_ROOT"] = pqid_root
    globals_dict["PROCESSED_DIR"] = processed_dir

    if globals_dict["METADATA_DESIGN_RUN_MODE"] == "smoke_test":
        globals_dict["LICENSE_REPORT_JSON_FILE"] = processed_dir / "pqid_license_governance_report_v3_smoke.json"
    else:
        globals_dict["LICENSE_REPORT_JSON_FILE"] = processed_dir / "pqid_license_governance_report_v3.json"

_metadata_notebook_bootstrap_license_preview()

license_report = json.loads(LICENSE_REPORT_JSON_FILE.read_text(encoding="utf-8"))

overview_rows = []
for section_name in [
    "license_category_counts",
    "distribution_rights_status_counts",
    "public_release_bucket_counts",
    "license_audit_priority_counts",
    "permission_response_status_counts",
    "manual_license_review_status_counts",
]:
    for key, value in license_report[section_name].items():
        overview_rows.append([section_name, key, value])
display(Markdown("### License Governance Overview\n\n" + markdown_table(["section", "value", "count"], overview_rows)))

breakdown_rows = []
for section_name in [
    "unresolved_no_license_by_validation_status",
    "unresolved_no_license_by_expected_model_stance",
    "unresolved_no_license_by_retrieval_strategy",
    "unresolved_no_license_by_source",
]:
    for key, value in license_report[section_name].items():
        breakdown_rows.append([section_name, key, value])
display(Markdown("### Unresolved No-License Breakdown\n\n" + markdown_table(["section", "value", "count"], breakdown_rows)))

top_repo_rows = []
for repo in license_report["top_unresolved_repositories"]:
    top_repo_rows.append([
        repo["repo"],
        repo["rows"],
        repo["validated_rows"],
        repo["generate_rows"],
        repo["repair_rows"],
        repo["robustness_compare_rows"],
        repo["priority_score"],
        repo["top_retrieval_strategy"],
    ])
display(Markdown(
    "### Top Unresolved Repositories\n\n" + markdown_table(
        [
            "repo",
            "rows",
            "validated_rows",
            "generate_rows",
            "repair_rows",
            "robustness_compare_rows",
            "priority_score",
            "top_retrieval_strategy",
        ],
        top_repo_rows,
    )
))
